# 🧮 01-数据结构与算法 · 主题 1：数组与哈希表

> 秋招算法第一课。数组是内存布局的基石，哈希表是「空间换时间」的核心武器。
> 本 notebook 目标：**看懂原理 → 能手写哈希表 → 吃透 9 道高频题 → 通过 25 道自测**。

## 学习路线

1. §2 数组：操作与复杂度公式
2. §3 哈希表：原理、冲突处理、均摊分析（含手写实现）
3. §4–§6 高频题精讲：两数之和 / 三数之和 / 移动零 / 轮转数组 / 合并区间 / 最长连续序列 / 除自身以外数组的乘积 / 和为 K 的子数组 / 多数元素
4. §7 复杂度总表
5. §8 常见 bug 与边界
6. §9 面试追问与扩展
7. §10 自测清单（L1/L2/L3）

**运行环境**：Python 3.8+，无第三方依赖，从上到下运行所有 cell 即可。

## §1 为什么数组与哈希是秋招地基

### 面试怎么考

- **数组**：负索引、切片、原地操作、双指针配合——大量题的基础载体
- **哈希表**：把 $O(n)$ 查找降到 $O(1)$，是「空间换时间」的典范，几乎每场面试必有 1 题
- **手撕题**：手写哈希表 / LRU / 去重——考察你是否真正理解哈希原理

### 数据规模 → 复杂度（30 秒判断法）

拿到题先看输入规模 $n$：

| $n$ | 允许的复杂度 | 典型做法 |
|------|-------------|---------|
| $n \le 20$ | $O(2^n)$ / $O(n!)$ | 暴力、回溯 |
| $n \le 10^3$ | $O(n^2)$ | 双循环 + 优化 |
| $n \le 10^5$ | $O(n \log n)$ | 排序、二分、堆 |
| $n \le 10^6+$ | $O(n)$ / $O(n \log n)$ | 一次扫描、哈希、排序 |

> 💡 **判断口诀**：1 秒约 $10^7$~$10^8$ 次基本操作，把复杂度代进 $n$，别超过这个量级。

### 心智模型

- 数组 = 一排编号的储物柜，**知道编号直接开柜**（$O(1)$ 随机访问）
- 哈希表 = 带「标签」的储物间，**把钥匙 hash 成柜号**，冲突了就在柜里排队（链地址）或找下一个空柜（开放寻址）

## §2 数组：操作与复杂度公式

### 2.1 随机访问 $O(1)$

数组在内存中**连续**存放，首地址为 $\text{base}$，元素大小 $s$ 字节：

$$\text{addr}(a_i) = \text{base} + i \times s$$

下标计算是一次乘法一次加法——常数时间，**与 $n$ 无关**。

### 2.2 插入 / 删除 $O(n)$

在第 $i$ 个位置插入需要把 $a_i \dots a_{n-1}$ 全部后移一位：

$$T_{insert}(i) = n - i \implies T_{insert} = O(n)$$

同理删除中间元素要前移。**只有尾部操作（append / pop）是 $O(1)$**（均摊，见 §7）。

### 2.3 Python 的 list 是什么

- Python `list` 是**动态数组**（不是链表）：底层连续数组 + 自动扩容
- 扩容策略：容量翻倍，均摊 $O(1)$
- 切片 `a[i:j]` 返回**新列表**，$O(k)$，$k = j - i$
- 负索引：`a[-1]` 等价于 `a[n-1]`，是语法糖，不是新数据结构

### 2.4 查找

- 按值查找：无序时 $O(n)$ 线性扫描；有序时 $O(\log n)$ 二分（主题 5 专讲）
- 哈希：$O(1)$——见 §3

In [ ]:
# 数组基础操作演示
arr = [1, 2, 3, 4, 5]

print('随机访问 arr[2] =', arr[2])       # O(1)

# 尾部操作 O(1)
arr.append(6)
print('append 后:', arr)

# 头部插入 O(n)：后续元素全部后移
arr.insert(0, 0)
print('insert(0) 后:', arr)

# 切片 O(k)，返回新列表
sub = arr[1:4]
print('arr[1:4] =', sub, '，原列表不变:', arr)

# 负索引语法糖
print('arr[-1] =', arr[-1], ' 等价于 arr[len(arr)-1] =', arr[len(arr) - 1])

# 按值查找
print('arr.index(3) =', arr.index(3), '，7 in arr =', 7 in arr)

# 推导式构造
squares = [x * x for x in range(6)]
print('squares =', squares)

## §3 哈希表：原理、冲突处理、均摊分析

### 3.1 哈希函数

把键 $k$ 映射到槽位：

$$h(k) = k \bmod m$$

要求：**计算快、分布均匀、确定性**。对字符串等复杂键，先用哈希算法把键变成整数再取模。

### 3.2 冲突与装载因子

不同键可能映射到同一槽（鸽巢原理：槽少于键数必冲突），两种主流处理：

**链地址法（chaining）**：每个槽一条链表，冲突键挂到同一条链上。

**开放寻址法（open addressing）**：冲突时按探测序列找下一个空位（线性探测 $h(k)+j$、二次探测、双重哈希）。

装载因子（load factor）：

$$\alpha = \frac{n}{m} \quad (n=\text{元素数},\ m=\text{槽数})$$

均匀哈希假设下：

- 链地址：查找期望 $E = 1 + \frac{\alpha}{2}$，$\alpha$ 控制在 $\le 1$ 则常数小
- 开放寻址（线性探测）：成功查找期望探测次数 $\approx \frac{1}{2}\left(1 + \frac{1}{1-\alpha}\right)$，$\alpha > 0.7$ 后性能急剧恶化

> ⚠️ **为什么 Python dict 用开放寻址而不用链地址**：缓存局部性好、无指针开销、内存紧凑；代价是删除要打「墓碑」（tombstone）标记。

### 3.3 resize 与均摊 $O(1)$

装载因子超阈值时翻倍扩容，重新哈希所有元素（单次 $O(n)$）：

$$\text{均摊成本} = \frac{n \times O(1) + O(n) \text{(扩容)}}{n} = O(1)$$

和动态数组扩容同理：每次扩容翻倍，总搬移成本 $n + \frac{n}{2} + \frac{n}{4} + \dots < 2n = O(n)$，均摊到 $n$ 次插入就是 $O(1)$。

### 3.4 Python 的可哈希性

- 不可变对象（int / str / tuple / frozenset）可哈希；可变对象（list / dict / set）不可哈希
- 自定义类需同时实现 `__hash__` 与 `__eq__`，且**相等的对象哈希值必须相同**

In [ ]:
class MyHashMap:
    '''链地址法手写哈希表：put / get / remove + 装载因子扩容'''
    def __init__(self, capacity=8):
        self.capacity = capacity
        self.table = [[] for _ in range(capacity)]
        self.size = 0

    def _hash(self, key):
        # Python 内置 hash：int 哈希为其自身，取模映射到槽位
        return hash(key) % self.capacity

    def put(self, key, value):
        bucket = self.table[self._hash(key)]
        for i, (k, v) in enumerate(bucket):
            if k == key:
                bucket[i] = (key, value)   # 键已存在：更新
                return
        bucket.append((key, value))        # 键不存在：新增
        self.size += 1
        if self.size > self.capacity * 0.75:
            self._resize(self.capacity * 2)

    def get(self, key, default=None):
        for k, v in self.table[self._hash(key)]:
            if k == key:
                return v
        return default

    def remove(self, key):
        bucket = self.table[self._hash(key)]
        for i, (k, v) in enumerate(bucket):
            if k == key:
                bucket.pop(i)
                self.size -= 1
                return True
        return False

    def _resize(self, new_capacity):
        old_table = self.table
        self.capacity = new_capacity
        self.table = [[] for _ in range(new_capacity)]
        for bucket in old_table:
            for k, v in bucket:
                self.table[self._hash(k)].append((k, v))


hm = MyHashMap()
for i in range(20):
    hm.put(i, i * 10)
print('插入 20 个键值对后: size =', hm.size, ', capacity =', hm.capacity)
print('get(5) =', hm.get(5))
print('get(99) =', hm.get(99, '未找到'))
hm.put(5, 999)
print('覆盖已有键 put(5, 999) 后 get(5) =', hm.get(5))
print('remove(5) =', hm.remove(5), ', 再 get(5) =', hm.get(5))

# 负数与字符串键
hm.put(-3, '负键')
hm.put('python', '字符串键')
print('get(-3) =', hm.get(-3), ', get(\'python\') =', hm.get('python'))

## §4 高频题精讲 A：两数之和 / 三数之和

### 题 1：两数之和（LeetCode 1，必背）

**题目**：给定数组与目标值 $target$，返回两数下标使 $a_i + a_j = target$。

**核心公式**：

$$a_j = target - a_i$$

**思路**：一次遍历，用哈希表记录「值 → 下标」。对每个 $a_i$，查 $target - a_i$ 是否已在表中——把 $O(n^2)$ 暴力降为 $O(n)$。

**复杂度**：时间 $O(n)$，空间 $O(n)$。

### 题 2：三数之和（LeetCode 15，去重是考点）

**题目**：找所有 $a_i + a_j + a_k = 0$ 的三元组，不能重复。

**思路**：排序 + 固定第一个数 + 双指针扫剩下区间。

**为什么去重要先排序**：排序后相同值相邻，跳过相同值即可避免重复三元组——用哈希去重还得额外 set 存结果。

**复杂度**：时间 $O(n^2)$，空间 $O(\log n)$（排序栈）。

> 💡 **变形题**：最接近的三数之和（16）、四数之和（18）——套路一样，排序 + 双指针。

In [ ]:
def two_sum(nums, target):
    # 公式：b = target - a，边遍历边查哈希
    seen = {}
    for i, x in enumerate(nums):
        b = target - x
        if b in seen:
            return [seen[b], i]
        seen[x] = i
    return []

print('test1:', two_sum([2, 7, 11, 15], 9))   # 期望 [0, 1]
print('test2:', two_sum([3, 3], 6))           # 期望 [0, 1]
print('test3:', two_sum([3, 2, 4], 6))        # 期望 [1, 2]
print('test4:', two_sum([1, 2, 3], 7))        # 期望 []

In [ ]:
def three_sum(nums):
    nums.sort()
    res = []
    n = len(nums)
    for i in range(n - 2):
        if i > 0 and nums[i] == nums[i - 1]:
            continue                 # 去重：同一数值只作第一个数一次
        if nums[i] > 0:
            break                    # 排序后首数>0，和不可能为 0
        lo, hi = i + 1, n - 1
        while lo < hi:
            s = nums[i] + nums[lo] + nums[hi]
            if s < 0:
                lo += 1
            elif s > 0:
                hi -= 1
            else:
                res.append([nums[i], nums[lo], nums[hi]])
                lo += 1
                hi -= 1
                while lo < hi and nums[lo] == nums[lo - 1]:
                    lo += 1          # 跳过重复的 lo
                while lo < hi and nums[hi] == nums[hi + 1]:
                    hi -= 1          # 跳过重复的 hi
    return res

print('test1:', three_sum([-1, 0, 1, 2, -1, -4]))  # 期望 [[-1,-1,2],[-1,0,1]]
print('test2:', three_sum([0, 0, 0]))               # 期望 [[0,0,0]]
print('test3:', three_sum([0, 1, 1]))               # 期望 []

## §5 高频题精讲 B：移动零 / 轮转数组 / 合并区间

### 题 3：移动零（LeetCode 283）

**题目**：原地把 0 移到末尾，保持非零元素相对顺序。

**思路**：快慢指针。`slow` 指向下一个非零元素应放的位置，`fast` 扫描：遇到非零就与 `slow` 交换。

**复杂度**：时间 $O(n)$，空间 $O(1)$。

### 题 4：轮转数组（LeetCode 189，三次反转法）

**题目**：数组右移 $k$ 位。

**关键**：先取 $k' = k \bmod n$（$k$ 可能大于 $n$）。

**三次反转公式**：

$$\text{rotate}(k) = \text{rev}(0, n-1) \circ \text{rev}(0, k'-1) \circ \text{rev}(k', n-1)$$

**为什么成立**：整体反转让尾部元素到头部，再各自反转恢复两段内部顺序。

**复杂度**：时间 $O(n)$，空间 $O(1)$。

### 题 5：合并区间（LeetCode 56）

**题目**：合并所有重叠区间。

**思路**：按左端点排序；若当前区间左端点 $\le$ 结果末尾右端点则合并（右端点取 max），否则追加。

**复杂度**：时间 $O(n \log n)$（排序主导），空间 $O(n)$（结果）。

**贪心正确性**：排序后区间按左端点单调不减，新区间只可能与结果最后一个区间重叠——贪心成立。

In [ ]:
def move_zeroes(nums):
    slow = 0
    for fast in range(len(nums)):
        if nums[fast] != 0:
            nums[slow], nums[fast] = nums[fast], nums[slow]
            slow += 1
    return nums

def rotate(nums, k):
    n = len(nums)
    k %= n                      # 关键：k 可能大于 n
    def rev(i, j):
        while i < j:
            nums[i], nums[j] = nums[j], nums[i]
            i += 1
            j -= 1
    rev(0, n - 1)
    rev(0, k - 1)
    rev(k, n - 1)
    return nums

print('move_zeroes:', move_zeroes([0, 1, 0, 3, 12]))          # 期望 [1,3,12,0,0]
print('rotate1:', rotate([1, 2, 3, 4, 5, 6, 7], 3))           # 期望 [5,6,7,1,2,3,4]
print('rotate2:', rotate([-1, -100, 3, 99], 2))               # 期望 [3,99,-1,-100]
print('rotate3(k>n):', rotate([1, 2, 3], 5))                  # 期望 [2,3,1]

In [ ]:
def merge(intervals):
    intervals.sort(key=lambda x: x[0])
    res = []
    for lo, hi in intervals:
        if res and lo <= res[-1][1]:
            res[-1][1] = max(res[-1][1], hi)   # 合并：更新右端点
        else:
            res.append([lo, hi])
    return res

print('test1:', merge([[1, 3], [2, 6], [8, 10], [15, 18]]))  # 期望 [[1,6],[8,10],[15,18]]
print('test2:', merge([[1, 4], [4, 5]]))                     # 期望 [[1,5]]
print('test3:', merge([[1, 4], [0, 4]]))                     # 期望 [[0,4]]
print('test4:', merge([]))                                   # 期望 []

## §6 高频题精讲 C：哈希五题（每题都是一类套路）

### 题 6：最长连续序列（LeetCode 128）

**题目**：无序数组里最长的连续数字序列长度（如 [100,4,200,1,3,2] → 4）。

**套路**：全部放进 set，**只从连续段的起点开始数**——若 $x-1$ 已在集合则跳过 $x$。

**为什么总复杂度是 $O(n)$**：每个元素最多被访问两次（一次被跳过、一次作为某段内部被数过），不是 $O(n^2)$。

### 题 7：除自身以外数组的乘积（LeetCode 238，禁止除法）

**题目**：返回 $b_i = \prod_{j \neq i} a_j$。

**公式（左前缀 × 右后缀）**：

$$b_i = \underbrace{(a_0 a_1 \cdots a_{i-1})}_{\text{left}[i]} \times \underbrace{(a_{i+1} \cdots a_{n-1})}_{\text{right}[i]}$$

**思路**：先从左往右算左前缀积（复用输出数组），再从右往左乘右后缀积——空间 $O(1)$（不计输出）。

### 题 8：和为 K 的子数组（LeetCode 560，前缀和套路）

**题目**：连续子数组和为 $k$ 的个数。

**公式**：子数组和 = 前缀和之差

$$\sum_{t=i}^{j} a_t = P_j - P_{i-1} \quad \Rightarrow \quad P_j - k = P_{i-1}$$

**思路**：遍历时维护前缀和 $P_j$，用哈希表统计「每个前缀和出现次数」，累加 $P_j - k$ 的次数。

**注意**：存在负数和 0 元素时不能用滑动窗口（单调性失效）——必须前缀和。

**补充：差分数组（区间加减，前缀和的逆运算）**：对区间 $[l, r]$ 统一加 $v$，只需 $diff[l] += v, \; diff[r+1] -= v$，最后对 $diff$ 求前缀和还原原数组——多次区间操作从 $O(nk)$ 降到 $O(n+k)$（典型题：LeetCode 370 区间加法）。

### 题 9：多数元素（LeetCode 169，Boyer-Moore 投票）

**题目**：出现次数 $> \lfloor n/2 \rfloor$ 的元素。

**思路**：候选者 + 票数。票数归零换候选者，不同元素互相抵消。

**证明**：众数数量超过其他所有元素之和，两两抵消后众数必然留下。

**复杂度**：时间 $O(n)$，空间 $O(1)$。

### 题 10：缺失的第一个正数（LeetCode 41，原地哈希套路）

**题目**：未排序整数数组中，返回缺失的最小正整数（如 [3,4,-1,1] → 2）。

**套路——用数组下标当哈希桶**：值 $v$ 若在 $[1, n]$，它的「正确归宿」是下标 $v-1$。遍历时把每个 $v$ 交换到 $nums[v-1]$；若 $v \le 0$ 或 $v > n$ 或已就位，直接跳过。一遍交换后扫一遍，找第一个 $nums[i] \ne i+1$ 的位置返回 $i+1$；全部就位则返回 $n+1$。

**关键点：空间 $O(1)$ 复用输入数组**——不另开哈希表，把原数组就地改造成「下标 → 值」的桶，用数组本身完成哈希表的存与查，省掉 $O(n)$ 额外空间。

**复杂度**：时间 $O(n)$（每个元素最多被交换一次到正确位置），空间 $O(1)$（不计输入）。


In [ ]:
def longest_consecutive(nums):
    s = set(nums)
    best = 0
    for x in s:
        if x - 1 in s:
            continue                # 不是起点，跳过
        length = 1
        while x + length in s:
            length += 1
        best = max(best, length)
    return best

print('test1:', longest_consecutive([100, 4, 200, 1, 3, 2]))           # 期望 4
print('test2:', longest_consecutive([0, 3, 7, 2, 5, 8, 4, 6, 0, 1]))   # 期望 9
print('test3:', longest_consecutive([1, 2, 0, 1]))                     # 期望 3
print('test4:', longest_consecutive([]))                               # 期望 0

In [ ]:
def product_except_self(nums):
    n = len(nums)
    res = [1] * n
    # 第一遍：res[i] = 左前缀积 a0*a1*...*a(i-1)
    for i in range(1, n):
        res[i] = res[i - 1] * nums[i - 1]
    # 第二遍：从右往左乘右后缀积
    right = 1
    for i in range(n - 1, -1, -1):
        res[i] *= right
        right *= nums[i]
    return res

print('test1:', product_except_self([1, 2, 3, 4]))       # 期望 [24, 12, 8, 6]
print('test2:', product_except_self([-1, 1, 0, -3, 3]))  # 期望 [0, 0, 9, 0, 0]
print('test3:', product_except_self([0, 0]))             # 期望 [0, 0]

In [ ]:
def subarray_sum(nums, k):
    # 前缀和：P_j - P_{i-1} = k  ⇔  统计 P_j - k 在之前出现几次
    count = 0
    prefix = 0
    seen = {0: 1}                # 前缀和为 0 出现 1 次（空前缀）
    for x in nums:
        prefix += x
        count += seen.get(prefix - k, 0)
        seen[prefix] = seen.get(prefix, 0) + 1
    return count

print('test1:', subarray_sum([1, 1, 1], 2))    # 期望 2
print('test2:', subarray_sum([1, 2, 3], 3))    # 期望 2
print('test3:', subarray_sum([0, 0, 0], 0))    # 期望 6
print('test4:', subarray_sum([-1, -1, 1], 0))  # 期望 1（[-1,1]）

In [ ]:
def majority_element(nums):
    # Boyer-Moore 投票：不同元素互相抵消，最后剩下的候选者即众数
    candidate = None
    votes = 0
    for x in nums:
        if votes == 0:
            candidate = x
        votes += 1 if x == candidate else -1
    return candidate

print('test1:', majority_element([3, 2, 3]))                  # 期望 3
print('test2:', majority_element([2, 2, 1, 1, 1, 2, 2]))      # 期望 2
print('test3:', majority_element([1]))                        # 期望 1

# 合法性校验：多数元素出现次数必须 > n//2
def check(nums):
    m = majority_element(nums)
    return nums.count(m) > len(nums) // 2
print('合法性校验:', check([2, 2, 1, 1, 1, 2, 2]), check([3, 2, 3]))

In [ ]:
def first_missing_positive(nums):
    """原地哈希：用数组下标当哈希桶，空间 O(1) 复用输入数组。
    nums[i] 的正确归宿是 nums[nums[i]-1]；值不在 [1, n] 的直接忽略。
    踩坑提醒：不要直接写 nums[i], nums[nums[i]-1] = nums[nums[i]-1], nums[i]
    ——元组赋值右边先求值、左边从左到右赋值，nums[i] 一变，
    nums[nums[i]-1] 里的下标就跟着错位，交换结果就错了。
    先存 v = nums[i]，让交换左右都引用稳定的 v；换回来的值下一轮再检查（i 不前进）。
    """
    n = len(nums)
    i = 0
    while i < n:
        v = nums[i]
        if 1 <= v <= n and nums[v - 1] != v:
            nums[i], nums[v - 1] = nums[v - 1], v   # v 已存好，下标与值都稳定
            # 换回来的值可能也要归位：i 不前进，下一轮继续处理当前位置
        else:
            i += 1
    for i in range(n):
        if nums[i] != i + 1:
            return i + 1
    return n + 1

# 题目要求的最小正整数
assert first_missing_positive([1, 2, 0]) == 3
assert first_missing_positive([3, 4, -1, 1]) == 2
assert first_missing_positive([7, 8, 9, 11, 12]) == 1
assert first_missing_positive([1]) == 2
assert first_missing_positive([2]) == 1
assert first_missing_positive([]) == 1
# 边界：全非正、大数、重复、已就位
assert first_missing_positive([-1, -5, 0]) == 1
assert first_missing_positive([10 ** 9]) == 1
assert first_missing_positive([1, 1]) == 2
assert first_missing_positive([2, 1]) == 3
print('缺失的第一个正数：全部断言通过')

## §7 复杂度总表（背熟）

### 数组 / list

| 操作 | 复杂度 | 说明 |
|------|--------|------|
| 随机访问 a[i] | $O(1)$ | 基址 + 偏移 |
| 尾部 append / pop | $O(1)$ 均摊 | 翻倍扩容均摊 |
| 头部 / 中间插入删除 | $O(n)$ | 元素搬移 |
| 按值查找（无序） | $O(n)$ | 线性扫描 |
| 切片 a[i:j] | $O(k)$ | 新列表拷贝 |
| sort() | $O(n \log n)$ | Timsort，稳定 |

### 哈希表（dict / set）

| 操作 | 复杂度 | 说明 |
|------|--------|------|
| 插入 / 删除 / 查找 | $O(1)$ 均摊 | 冲突退化 $O(n)$ |
| 遍历 | $O(n)$ | Python 3.7+ 保持插入顺序 |
| 扩容 resize | 单次 $O(n)$，均摊 $O(1)$ | 翻倍 + rehash |

### 均摊分析（为什么翻倍扩容均摊 $O(1)$）

插入 $n$ 个元素，扩容总搬移成本：

$$n + \frac{n}{2} + \frac{n}{4} + \cdots < 2n = O(n)$$

$$\text{均摊每次插入} = \frac{O(n)}{n} = O(1)$$

> 💡 **hash 冲突退化**：若哈希函数固定且攻击者构造同余键，单桶链长可达 $O(n)$——Python 3.3+ 用随机盐（PYTHONHASHSEED）防哈希碰撞攻击。

## §8 常见 bug 与边界（面试踩坑区）

- **空数组 / 单元素**：`nums[0]` 越界——先判 `len(nums)`
- **负下标陷阱**：`nums[-1]` 是最后一个元素，不是「倒数第二个」
- **k 大于 n**：轮转、二分等一律先 `k %= n`
- **哈希键必须不可变**：`dict[list]` 直接 TypeError；tuple 可作键
- **重复元素**：两数之和返回**下标**，同一元素不能复用两次——先查哈希再写入，保证不误用自己
- **整数溢出**：Python 无溢出；C++/Java 里 `a + b` 可能溢出（用 C++ 面试时注意）
- **前缀和别忘 `{0: 1}`**：和为 K 的子数组最容易漏掉「从头开始的子数组」
- **三数之和去重顺序**：排序后跳过相邻重复值，而不是最后 set 去重
- **原地操作 vs 返回值**：看清题目要求（移动零是否返回）
- **list 的 in 是 $O(n)$**：set / dict 的 in 才是 $O(1)$——别混用

> ⚠️ **最隐蔽的一个**：`for i in range(len(nums))` 里同时 `nums.pop(i)` 会导致索引错位——要删元素时倒序遍历或先收集下标。

## §9 面试追问与扩展（加分项）

### 9.1 大数据场景：内存装不下怎么办

- **去重**：布隆过滤器（Bloom Filter）——位数组 + $k$ 个哈希函数，**允许假阳性、无假阴性**
- **频次统计**：Count-Min Sketch 近似计数
- **精确去重**：外部排序 + 归并

### 9.2 Python dict 实现细节（高频八股）

- Python 3.6+ 保持插入顺序（3.7 起为语言规范）；3.11 起紧凑 dict 节省内存
- 冲突处理：开放寻址 + 伪随机探测（随机盐防哈希碰撞攻击）
- dict vs set：底层都是哈希表；dict 存「键→值」，set 存「键集合」
- 面试手写时用普通 dict + `get` 即可，能讲清原理更加分

### 9.3 手撕题预告（后续主题覆盖）

- LRU 缓存（哈希 + 双向链表）→ 链表主题
- 前缀树 Trie（哈希的树形推广）→ 树主题
- 字符串哈希 / 滚动哈希 → 字符串主题

## §10 自测清单（L1 必会 / L2 进阶 / L3 挑战）

> 每题先自己写，再看答案思路。会写 = 能讲清复杂度 + 边界。

### L1 必会（10 题）

- [ ] 1. 两数之和（LeetCode 1）
- [ ] 2. 移动零（283）
- [ ] 3. 合并两个有序数组（88）
- [ ] 4. 多数元素（169）
- [ ] 5. 只出现一次的数字（136）
- [ ] 6. 有效的字母异位词（242）
- [ ] 7. 轮转数组（189）
- [ ] 8. 找出数组交集（349）
- [ ] 9. 存在重复元素（217）
- [ ] 10. 买卖股票的最佳时机（121，贪心/DP）

### L2 进阶（10 题）

- [ ] 11. 三数之和（15）
- [ ] 12. 最长连续序列（128）
- [ ] 13. 除自身以外数组的乘积（238）
- [ ] 14. 和为 K 的子数组（560）
- [ ] 15. 合并区间（56）
- [ ] 16. 字母异位词分组（49）
- [ ] 17. 颜色分类（75）
- [ ] 18. 最大子数组和（53）
- [ ] 19. 螺旋矩阵（54）
- [ ] 20. 下一个排列（31）

### L3 挑战（5 题）

- [ ] 21. 缺失的第一个正数（41）
- [ ] 22. 最大矩形（85）
- [ ] 23. 和为 K 的最长子数组（面试手撕）
- [ ] 24. 快照数组（1146）
- [ ] 25. 网格照明（1001）

### 自测要点

- 每题能写出**最优复杂度**解法 + 一个反例验证
- 讲清「为什么哈希能把 $O(n^2)$ 变 $O(n)$」——用「空间换时间」一句话概括
- 手写链地址哈希表 5 分钟内完成（含扩容）

## 附录：本节参考与下节预告

### 参考

- 《算法导论》第 11 章：哈希表（均匀哈希、装载因子、开放寻址）
- Python 官方文档：dict 实现说明
- LeetCode 分类题单：数组 / 哈希表

### 下节预告：主题 2 — 双指针与滑动窗口

数组题的两大套路：

- **双指针**：相向（盛水容器 / 接雨水）、同向快慢（移动零 / 去重）
- **滑动窗口**：维护窗口左右边界，配合哈希记录窗口内容——无重复最长子串（3）、最小覆盖子串（76）

> 💡 本主题验收：`高频题单.md` 中「数组/哈希」一节全部打勾，模板代码 ≥ 2 份。